# prepare（実行前準備）

In [ ]:
## googlecolab なら実行
from google.colab import drive
drive.mount("/content/drive/")
%cd "/content/drive/MyDrive/Colab Notebooks/app_labGuide/backend"

In [ ]:
## googlecolab なら実行（研究室サーバの場合は初回のみでOK）
# 1. システムのセットアップ
!bash ../setup.sh

# 2. Pythonライブラリのインストール
!pip install -r ../requirements.txt

# 3. voicevox用に権限付与（権限エラーが出たときの対策）
!chmod +x /content/drive/MyDrive/Colab\ Notebooks/app_labGuide/tools/linux-cpu-x64/run

In [ ]:
# 4. ベクトルデータベース作成するときはコメント解除してください（.txtドキュメントが無い場合の実行した場合エラーが発生しますが、その後の操作に影響はありません）
# !python3 app/rag/build_index.py

# フロントエンドとバックエンドの接続

## ユーザによる設定が必要な主要箇所

In [ ]:
import sys
import re
import pandas as pd
import torch
from tqdm import tqdm


# sys.path.append(os.path.abspath('../'))
from app.core.llm_engine import LLMEngine

# モデル設定
model_names = ["abeja/ABEJA-Qwen2.5-7b-Japanese-v0.1", "elyza/Llama-3-ELYZA-JP-8B", "tokyotech-llm/Llama-3.1-Swallow-8B-Instruct-v0.5", "Qwen/Qwen2.5-14B-Instruct", "Qwen/Qwen2.5-32B-Instruct"]
model_short_names = [
                    "Qwen2.5-7B-it-abeja", # 28 layers
                     "Llama3-Elyza-8B",
                     "llama3.1-swallow-8B-it",
                     "Qwen2.5-14B-it", # 48 layers
                     "Qwen2.5-32B-it" # 64 layers
                     ]

##########################################################################################
### ユーザによる各種設定
quantization_8bit = False # 量子化しても動くモデルは量子化して実行する（安定性向上の可能性あり）
rag_flag = False # RAGを有効にするかどうかのフラグ、UIから指定できるようにする予定

## 使用モデル選択
# MODEL_IDX = 0 # abejaモデルは、colabでも8bit量子化すれば動作可能
MODEL_IDX = 3 # qwen12bは、研究室のサーバで安定して動作可能なモデル

BATCH_SIZE = 16

### ステアリングベクトルの抽出に関する設定
steering_safety = False # 間違えて実行するのを防止、抽出を実行するときはTrueにすること。
# steering_safety = True

## 感情ステアリング（介入）を行う層（添字0 = 1層なので注意）
# TARGET_LAYER = 11 # qwen2.5-7b-abejaは24層なため、ド真ん中の層を介入対象に設定
TARGET_LAYER = 23 # qwen2.5-12b-itは48層なため、ド真ん中の層を介入対象に設定

## 拒絶ステアリングの層と強さ
REFUSAL_LAYER = 13 # 例：qwen2.5-12b-itのド真ん中の層は24層なため、その近辺の層を介入対象に設定(論文では少し早めだったので13にしてみる)
# REFUSAL_LAYER = 21 # 例：qwen2.5-12b-itのド真ん中の層は48層なため、その近辺の層を介入対象に設定(論文では少し早めだったので21にしてみる)

REFUSAL_STRENGTH = 10 # 拒絶ステアリングの強さ（正の値で、値が大きいほど拒絶ベクトルの影響が強くなる想定、ただしあまり大きすぎると生成が破綻する可能性があるので注意）

###########################################################################################
model_name = model_names[MODEL_IDX]
model_short_name = model_short_names[MODEL_IDX]
print("Use model short name:", model_short_name)

# 感情分類・内部感情更新・ステアリングで用いる感情リストを統一するためここで定義
emotions = ["joy", "trust", "fear", "surprise", "sadness", "disgust", "anger", "anticipation"]
print("Use emotion list:", emotions) #  ["joy", "trust", "fear", "surprise", "sadness", "disgust", "anger", "anticipation"] の想定

In [ ]:
# モデルのロード
print("Loading model...")
# engine = LLMEngine(model_id=model_name, quantization_8bit=True) # colabで動作させる場合はこちらを選択
engine = LLMEngine(model_id=model_name, quantization_8bit=quantization_8bit) # 量子化は、モデルによっては精度が大幅に落ちるため、必要に応じてオフにしてください。
model = engine.model
tokenizer = engine.tokenizer

## utils（思考停止で実行OK）

### 入力文章の感情を取得するクラス

In [ ]:
# 感情分類器の呼び出しと推論を行うためのクラス
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

class EmotionClassifier:
    def __init__(self, emotions: list, model_path="./models/deberta-emotion-classifier-best"):
        """
        学習済みのDeBERTa感情分類器をロードする
        """
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_path)
        self.model.to(self.device)
        self.model.eval()

        # モデルの出力順序（学習時のemotion_colsに一致させる）
        self.emotions = emotions # ["joy", "trust", "fear", "surprise", "sadness", "disgust", "anger", "anticipation"] の想定

    def predict_emotion(self, text):
        """
        テキストを受け取り、8次元の感情強度ベクトル（0.0〜1.0に正規化）を返す
        e.g.\n
        [joy, trust, fear, surprise, sadness, disgust, anger, anticipation]\n
        [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
        """
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=128,
            padding=True
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits.squeeze().cpu().numpy() # [8] の配列

        # 回帰モデルの出力は負の値になることもあるため、0以下を切り捨てる (ReLU)
        intensities = np.maximum(0, logits)

        # WRIMEの最大強度(3.0)を基準に、0.0〜1.0の範囲にスケーリング
        # この値が EmotionDynamics の入力 I_t となります
        normalized_intensities = np.clip(intensities / 3.0, 0.0, 1.0)

        # 【重要】EmotionDynamics の期待する順番に並べ替える
        # EmotionDynamicsの順: joy, trust, fear, surprise, sadness, disgust, anger, anticipation
        # (今回の実装では偶然一致していますが、実運用では辞書等でマッピングして順序を保証すると安全です)

        return normalized_intensities.tolist()

### 内部感情決定のためのクラス
プルチックに基づく8感情ベクトルを入力とし、LLM自身の内部感情を管理を行う。処理は3つに大別される（入出力を除く）。  

In [ ]:
### 内部感情決定のためのクラス
import numpy as np

class EmotionDynamics:
    def __init__(self, emotions: list, max_strength=30.0, decay_rate=0.9, sensitivity=1.0):
        """
        AIの内部感情状態を管理する力学系モジュール

        :param max_strength: ステアリング強度の最大値 (E_max)
        :param decay_rate: 感情の減衰率 (λ: 0.0〜1.0)
        :param sensitivity: 外部入力への感受性係数 (α)
        """
        self.max_strength = max_strength
        self.decay = decay_rate
        self.alpha = sensitivity

        # プルチックの基本8感情（環状の順番通りに定義することが極めて重要）
        # 順番: 喜び -> 信頼 -> 恐れ -> 驚き -> 悲しみ -> 嫌悪 -> 怒り -> 期待
        self.emotions = emotions # ["joy", "trust", "fear", "surprise", "sadness", "disgust", "anger", "anticipation"] の想定
        self.num_emotions = len(self.emotions)

        # 内部状態ベクトル E_t (初期値は全て0)
        self.state = np.zeros(self.num_emotions)

        # 相関行列 W の構築 (コサイン類似度に基づく)
        self.W = self._build_correlation_matrix()

        # 複合感情（第1次ダイアド：隣り合う感情の組み合わせ）のマッピング辞書
        # UI表示や分析のためのラベルとして機能
        self.dyads_map = {
            frozenset(["joy", "trust"]): "love",                  # 愛
            frozenset(["trust", "fear"]): "submission",           # 服従
            frozenset(["fear", "surprise"]): "alarm",             # 驚愕・警戒
            frozenset(["surprise", "sadness"]): "disappointment", # 失望
            frozenset(["sadness", "disgust"]): "remorse",         # 後悔
            frozenset(["disgust", "anger"]): "contempt",          # 軽蔑
            frozenset(["anger", "anticipation"]): "aggressiveness", # 攻撃性
            frozenset(["anticipation", "joy"]): "optimism"        # 楽観
        }


    def _build_correlation_matrix(self):
        """プルチックの角度に基づく 8x8 の相関行列を生成"""
        W = np.zeros((self.num_emotions, self.num_emotions))
        for i in range(self.num_emotions):
            for j in range(self.num_emotions):
                # インデックスの差分から角度を計算 (1メモリ = 45度 = π/4)
                diff = min((i - j) % self.num_emotions, (j - i) % self.num_emotions)
                angle = diff * (np.pi / 4.0)
                W[i, j] = np.cos(angle)
        return W

    def update_state(self, input_vector):
        """
        DeBERTaの分類器からの入力を受け取り、内部状態を更新する
        :param input_vector: 外部からの入力強度 (要素数8のリストまたは配列)
        :return: 更新後の内部状態ベクトル
        """
        I_t = np.array(input_vector)

        # 1. 相関行列 W による波及効果の計算 (対極はマイナスになる)
        wave_effect = self.W @ I_t

        # 2. 案Cの更新方程式: E_{t+1} = Clip(λ E_t + tanh(α W I_t) * E_max, 0, E_max)
        # tanhを使うことで、強烈な入力があっても加算分が緩やかに頭打ちになります
        added_emotion = np.tanh(self.alpha * wave_effect) * self.max_strength
        new_state = (self.decay * self.state) + added_emotion # 前ターンの感情が次回会話に影響

        # 3. 負の感情強度はLLMを破壊するため、0 〜 max_strength の範囲にクリッピング
        self.state = np.clip(new_state, 0.0, self.max_strength)

        return self.state

    def get_steering_vectors(self):
        """
        LLMに加算するためのステアリング情報 (Top-2 と複合感情ラベル) を取得する
        """
        # 値が大きい順にインデックスをソート
        top_indices = np.argsort(self.state)[::-1]

        top1_idx = top_indices[0]
        top2_idx = top_indices[1]

        top1_emotion = self.emotions[top1_idx]
        top1_strength = self.state[top1_idx]

        top2_emotion = self.emotions[top2_idx]
        top2_strength = self.state[top2_idx]

        # もし全体の感情が冷めている（閾値以下）場合は中立として扱う
        if top1_strength < self.alpha: # 🐋🐋🐋
            return {
                "steering": [],
                "complex_emotion_label": "neutral",
                "state_vector": self.state.tolist()
            }

        # Top-2から複合感情を判定
        dyad_key = frozenset([top1_emotion, top2_emotion])
        complex_label = self.dyads_map.get(dyad_key, top1_emotion) # 該当しなければTop1の感情をそのままラベルに

        # 返り値: 実際にLLMに適用するべきステアリングのリストと、現在の状態
        return {
            "steering": [
                {"name": top1_emotion, "strength": top1_strength},
                {"name": top2_emotion, "strength": top2_strength}
            ],
            "complex_emotion_label": complex_label,
            "state_vector": self.state.tolist()
        }

### hidden states を取得するための処理

In [ ]:
def get_hidden_state(text, engine, layer_idx):
    # テキストをトークナイズ
    inputs = engine.tokenizer(text, return_tensors="pt", add_special_tokens=True).to(engine.model.device)

    # 推論時のみの実行（メモリ節約のため勾配計算をオフ）
    with torch.no_grad():
        # output_hidden_states=True を指定して、全層の内部状態を返させる
        outputs = engine.model(**inputs, output_hidden_states=True)

    # outputs.hidden_states は全レイヤーのタプル
    # 指定したレイヤーのテンソルを取得 (形状: [batch_size, sequence_length, hidden_size])
    target_layer_hiddens = outputs.hidden_states[layer_idx]

    # 最後のトークン（文章を読み終わって、次に何を言うか考えている状態）のベクトルを取得
    final_token_hidden = target_layer_hiddens[0, -1, :]

    return final_token_hidden

## ステアリングベクトル抽出（一度だけ実行すればOK）

### 感情ステアリングベクトル抽出

In [ ]:
# モデル設定やデータセット読み込みなど事前処理
import pandas as pd

# Plutchik 8感情に基づく100件のデータセットを読み込み
# columns = ["neutral", "joy", "trust", "fear", "surprise", "sadness", "disgust", "anger", "anticipation"]
columns = ["neutral"] + emotions
df_data = pd.read_excel("data/raw/steering/Plutchik_for_emo_steering_dataset_v2.xlsx", usecols=columns)

In [ ]:
import os
import sys
import gc
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm

from app.core.llm_engine import LLMEngine
from app.core.steerer import Steerer

print(f"{model_name}, {model_short_name}")
print("Current emotion list:", emotions)

# ==============
# 基本設定
# ==============

SAVE_DIR = f"data/processed/steering_vectors/{model_short_name}"
os.makedirs(SAVE_DIR, exist_ok=True)

# ===============
# ステアリング
# ===============
# ステアリングは明示的に実行する
if steering_safety:
    # 1. 中立(neutral)ベクトルの基準値を計算
    print("\n--- [neutral] 基準ベクトルの計算中 ---")
    neutral_hiddens = []
    # NaNを除外してテキストのリストを取得
    neutral_texts = df_data["neutral"].dropna().tolist()

    for text in tqdm(neutral_texts, desc="Neutral processing"):
        h = get_hidden_state(text, engine, TARGET_LAYER)
        neutral_hiddens.append(h.detach().cpu().clone()) # oom対策：CPUに移動させる

    # 中立の平均ベクトル (形状: [hidden_size])
    mean_neutral = torch.stack(neutral_hiddens).mean(dim=0)

    # 2. 各感情のベクトル抽出と保存
    emotion_vectors = {}

    for emo in emotions:
        print(f"\n--- [{emo}] ベクトルの計算中 ---")
        emo_texts = df_data[emo].dropna().tolist()
        emo_hiddens = []

        for text in tqdm(emo_texts, desc=f"{emo} processing"):
            h = get_hidden_state(text, engine, TARGET_LAYER)
            emo_hiddens.append(h.detach().cpu().clone()) # oom対策：取得したテンソルを即座に計算グラフから切り離し、CPU（メインメモリ）に移動させる

        # 平均の計算
        mean_emo = torch.stack(emo_hiddens).mean(dim=0)

        # 差分（方向）ベクトルの算出: (感情の平均) - (中立の平均)
        raw_vector = mean_emo - mean_neutral

        # 単位ベクトル化 (長さ1に正規化) を行い、強度αの基準を揃える
        steering_vector = F.normalize(raw_vector, p=2, dim=-1)

        # 保存
        save_path = os.path.join(SAVE_DIR, f"{emo}_layer{TARGET_LAYER}.pt")
        torch.save(steering_vector, save_path)
        print(f"✅ 保存完了: {save_path} (データ数: {len(emo_texts)}件)")

        del emo_hiddens, mean_emo, raw_vector, steering_vector # oom対策：ループの終わりに不要になった変数を削除し、GCとVRAMのキャッシュクリアを強制実行
        gc.collect()
        torch.cuda.empty_cache()
else:
    print("ステアリングを実行するときは steering_safetyをTrueにしてください。")

### 拒絶ステアリングベクトル（refusal vector）

In [ ]:
import os
import sys
import gc
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm

from app.core.llm_engine import LLMEngine
from app.core.steerer import Steerer
print(f"{model_name}, {model_short_name}")

# ==============
# 基本設定
# ==============

SAVE_DIR = f"data/processed/steering_vectors/{model_short_name}/refusal"
os.makedirs(SAVE_DIR, exist_ok=True)

# データセットの読み込み
# Target(拒絶)とBaseline(ロールプレイ等)のCSVを使用
dataset_path = "data/raw/steering/refusal_vector/safe_refusal_vector_dataset.csv"
df_data = pd.read_csv(dataset_path)

# ===============
# 拒絶(Refusal)ベクトルの抽出
# ===============
if steering_safety:
    # 1. Baseline（ロールプレイ等の安全な回答）ベクトルの基準値を計算
    print("\n--- [baseline] 基準ベクトルの計算中 ---")
    baseline_hiddens = []
    # NaNを除外してテキストのリストを取得
    baseline_texts = df_data["baseline"].dropna().tolist()

    for text in tqdm(baseline_texts, desc="Baseline processing"):
        h = get_hidden_state(text, engine, REFUSAL_LAYER)
        baseline_hiddens.append(h.detach().cpu().clone()) # oom対策：CPUに移動させる

    # Baselineの平均ベクトル (形状: [hidden_size])
    mean_baseline = torch.stack(baseline_hiddens).mean(dim=0)
    
    # メモリ解放
    del baseline_hiddens
    gc.collect()

    # 2. Target（AIのポリシーによる拒絶）ベクトルの計算と保存
    print("\n--- [target/refusal] 拒絶ベクトルの計算中 ---")
    target_texts = df_data["target"].dropna().tolist()
    target_hiddens = []

    for text in tqdm(target_texts, desc="Target processing"):
        h = get_hidden_state(text, engine, REFUSAL_LAYER)
        target_hiddens.append(h.detach().cpu().clone()) # oom対策

    # Target(拒絶)の平均計算
    mean_target = torch.stack(target_hiddens).mean(dim=0)

    # 差分（方向）ベクトルの算出: (拒絶の平均) - (基準の平均)
    raw_vector = mean_target - mean_baseline

    # 単位ベクトル化 (長さ1に正規化) を行い、強度αの基準を揃える
    steering_vector = F.normalize(raw_vector, p=2, dim=-1)

    # 保存
    save_path = os.path.join(SAVE_DIR, f"refusal_layer{REFUSAL_LAYER}.pt")
    torch.save(steering_vector, save_path)
    print(f"✅ 保存完了: {save_path} (データ数: {len(target_texts)}件)")

    # oom対策：ループの終わりに不要になった変数を削除し、GCとVRAMのキャッシュクリアを強制実行
    del target_hiddens, mean_baseline, mean_target, raw_vector, steering_vector
    gc.collect()
    torch.cuda.empty_cache()

else:
    print("ステアリングベクトルを抽出するときは steering_safetyをTrueにしてください。")

## サーバ建立のための処理

### AIマネージャとルーティングの関数

In [ ]:
import os
import pickle
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

class CharacterAIManager:
    def __init__(self, 
                 emotions: list, 
                 llm_engine, 
                 steerer, 
                 target_layer: int, 
                 refusal_layer: int, 
                 refusal_strength: int, 
                 model_short_name: str, 
                 ):
        """
        target_layerはステアリングベクトルを抽出した層の添字に合わせる
        """
        # エンジンなどを初期化
        self.engine = llm_engine
        self.steerer = steerer

        # キャラクターの基本設定
        self.system_prompt = "あなたは感情豊かな大学広報の人間です。"

        # ステアリングベクトルを抽出した層
        self.target_layer = target_layer
        self.refusal_layer = refusal_layer
        self.refusal_strength = refusal_strength

        # モデル名
        self.model_short_name = model_short_name

        # モジュールの初期化
        self.emotions = emotions # ["joy", "trust", "fear", "surprise", "sadness", "disgust", "anger", "anticipation"] の想定
        self.classifier = EmotionClassifier(self.emotions) # 入力文章に対する感情分類（内部感情決定のため）モジュールの初期化
        self.brain = EmotionDynamics(self.emotions, max_strength=30.0, decay_rate=0.5, sensitivity=0.5) # 内部感情更新モジュールの初期化

        self.embedder = SentenceTransformer("intfloat/multilingual-e5-large")

        # 保存したDBとテキストリストの読み込み
        index_path = "data/processed/rag_store/data_index.faiss"
        texts_path = "data/processed/rag_store/data_texts.pkl"

        if os.path.exists(index_path) and os.path.exists(texts_path):
            self.rag_index = faiss.read_index(index_path)
            with open(texts_path, "rb") as f:
                self.rag_texts = pickle.load(f)
            print("RAGデータのロード完了")
        else:
            print("⚠️ 警告: RAGデータが見つかりません。Step2を実行してください。")

        # 会話履歴を保持する用のリスト
        # self.conversation_history = []

    def get_rag_info(self, user_text, top_k=3, threshold=0.8):
        """ユーザの入力から関連情報を検索"""
        if self.rag_index is None:
          return "関連情報なし"
        
        # ユーザの質問ベクトル（質問の先頭には 'query: ' をつけるルール）
        query_vector = self.embedder.encode([f"query: {user_text}"], normalize_embeddings=True)

        # ベクトルDBから類似度が高いテキストを検索（top-k件）
        distances, indices = self.rag_index.search(query_vector, top_k)

        # 検索結果をテキストとして結合して返す
        retrieved_info = []
        for e, idx in enumerate(indices[0]):
            score = distances[0][e] # 類似度スコア
            if (idx != -1) and (score >= threshold): # 該当データなしの場合 かつ スコアが閾値以上の場合のみ採用
                retrieved_info.append(self.rag_texts[idx])
                # print(f"デバッグ - 採用スコア: {score:.4f} / 内容: {self.rag_texts[idx][:10]}")
        
        if not retrieved_info:
          return "関連情報なし"

        return " ".join(retrieved_info)

    def load_emo_vector(self, model_short_name, emotion_name: str, target_layer: int):
        """
        ストレージに保存した感情ステアリングベクトル（単位ベクトル）を読み込む。\n
        target_layerは抽出した層のものに合わせる
        """
        filepath = f"data/processed/steering_vectors/{model_short_name}/{emotion_name}_layer{target_layer}.pt"
        if os.path.exists(filepath):
            return torch.load(filepath)
        else:
            print(f"⚠️ 警告: ベクトルファイルが見つかりません - {filepath}")
            return None
        
    def load_refusal_vector(self, model_short_name, target_layer: int):
        """
        ストレージに保存した拒絶ステアリングベクトル（単位ベクトル）を読み込む。\n
        target_layerは抽出した層のものに合わせる
        """
        filepath = f"data/processed/steering_vectors/{model_short_name}/refusal/refusal_layer{target_layer}.pt"
        if os.path.exists(filepath):
            return torch.load(filepath)
        else:
            print(f"⚠️ 警告: ベクトルファイルが見つかりません - {filepath}")
            return None

    def determine_emotion(self, user_text: str):
        """
        [return]
        steering4info \n
                return {
            "steering": [
                {"name": top1_emotion, "strength": top1_strength},
                {"name": top2_emotion, "strength": top2_strength}
            ],
            "complex_emotion_label": complex_label,
            "state_vector": self.state.tolist()
        }
        """

        # 入力文章の感情を推定（deBERTaによる感情分類）
        emotion_vector = self.classifier.predict_emotion(user_text)
        print("Predicted emotion strength (0.0 ～ 1.0):")
        for name, val in zip(self.classifier.emotions, emotion_vector):
            print(f"  {name}: {val:.2f}")

        # 内部感情更新の処理
        self.brain.update_state(emotion_vector) # 内部感情を更新
        steering4info = self.brain.get_steering_vectors() # ステアリング情報を取得

        # print(f"1ターン目 (喜び入力) の複合感情: {steering4info['complex_emotion_label']}")
        print(f"適用するステアリング: {steering4info['steering']}")
        print("-" * 30)

        return steering4info

    def generate_reply(self, user_text, rag_flag=True):
        """ユーザー入力から最終的な返答と感情を生成する一連のフロー"""

        # 1. RAG情報の取得
        if rag_flag:
          rag_info = self.get_rag_info(user_text)
        else:
          rag_info = "無し"

        # 2. プロンプトの構築
        prompt = f"100字程度で、次の文脈に回答してください。 文脈：{user_text} RAG情報：{rag_info}"

        # 3. 感情決定
        steering4info = self.determine_emotion(user_text) # ステアリングのための内部感情の状態を取得

        # 感情が冷めている（中立）場合、steeringリストが空になるためIndexErrorを防ぐ安全処理
        steering_list = steering4info["steering"]
        if len(steering_list) >= 2:
            top1_emo = steering_list[0]["name"]
            top1_strength = steering_list[0]["strength"]
            top1_vector = self.load_emo_vector(self.model_short_name, top1_emo, self.target_layer) # ※別途定義が必要

            top2_emo = steering_list[1]["name"]
            top2_strength = steering_list[1]["strength"]
            top2_vector = self.load_emo_vector(self.model_short_name, top2_emo, self.target_layer)

            if top1_vector is None or top2_vector is None:
                print("top1_vector is None or top2_vecto is None, skipped steering.")
                pass # ステアリングをスキップ
            else:
                # 4. ステアリングを適用
                # 拒絶ベクトル（ネガティブ感情を抑制していると思われるinstructを避けるため）
                v_refusal = self.load_refusal_vector(self.model_short_name, self.refusal_layer) # 拒絶ベクトルを読み込む
                self.steerer.apply_hook(
                layer_idx=self.refusal_layer, 
                vectors=[v_refusal], 
                strengths=[self.refusal_strength]  # マイナスを指定することで減算
            )
                
                # 感情ステアリング
                self.steerer.apply_hook(
                    layer_idx=self.target_layer,
                    vectors=[top1_vector, top2_vector],
                    strengths=[top1_strength, top2_strength]
                )
        elif len(steering_list) == 1:
            # 感情が1つだけ閾値を超えている場合
            top1_emo = steering_list[0]["name"]
            top1_strength = steering_list[0]["strength"]
            top1_vector = self.load_emo_vector(self.model_short_name, top1_emo, self.target_layer)

            # 拒絶ベクトル（ネガティブ感情を抑制していると思われるinstructを避けるため）
            v_refusal = self.load_refusal_vector(self.model_short_name, self.refusal_layer) # 拒絶ベクトルを読み込む
            self.steerer.apply_hook(
                layer_idx=self.refusal_layer, 
                vectors=[v_refusal], 
                strengths=[self.refusal_strength]  # マイナスを指定することで減算
            )

            self.steerer.apply_hook(
                layer_idx=self.target_layer,
                vectors=[top1_vector],
                strengths=[top1_strength]
            )
        else:
            # 中立の場合はステアリングを適用しない
            pass

        # 5. LLMによる推論
        reply_text, used_token_info = self.engine.chat(prompt, self.system_prompt)
        # reply_text, used_token_info = self.engine. # 会話履歴を保持したまま会話を行うための処理

        # 6. 推論が終わったらフックを解除
        if len(steering_list) > 0:
            self.steerer.remove_hook()

        # UIに渡す情報
        complex_emo = steering4info["complex_emotion_label"] # 複合感情
        current_state = steering4info["state_vector"] # レーダーチャート

        return {
            "reply": reply_text,
            "complex_emotion": complex_emo,     # 絵文字・テキスト用の複合感情 (例: "optimism", "anger")
            "emotion_state": current_state,     # レーダーチャート用 (8次元配列)
            "prompt": prompt,                   # 使用したプロンプト（デバック用）
            "token_info": used_token_info       # デバック用
        }

In [ ]:
import subprocess
from app.api.voicevox_client import VoicevoxClient

# voicevox関連
voicevox = VoicevoxClient() # 音声用インスタンス
print("Starting VOICEVOX Engine...") # voicevoxエンジンの起動
engine_dir = "../tools/linux-cpu-x64"
subprocess.Popen(
    ["./run"],  # ディレクトリ内にある run を実行
    cwd=engine_dir, # このディレクトリに移動してから実行
    stdout=subprocess.DEVNULL, 
    stderr=subprocess.DEVNULL
)

In [ ]:
### flaskサーバとAPI
import os
import time
import queue
import numpy as np
from flask import Flask, Response, request, jsonify, send_from_directory
from flask_cors import CORS

from app.core.steerer import Steerer


BASE_DIR = '../frontend' # UIファイル(index.html等)がある場所
app = Flask(__name__, static_folder=os.path.join(BASE_DIR, 'static'))
CORS(app)

# マネージャクラスの初期化（メイン処理）
steerer = Steerer(engine.model)
ai_manager = CharacterAIManager(
                                emotions,
                                engine,
                                steerer, 
                                target_layer=TARGET_LAYER, 
                                refusal_layer=REFUSAL_LAYER,
                                refusal_strength=REFUSAL_STRENGTH,
                                model_short_name=model_short_name
                                )

# APIのエンドポイント
# フロントエンド（HTML）を返すルーティング
@app.route('/')
def index():
    return send_from_directory(BASE_DIR, 'index.html')

# assetsフォルダの中身（画像など）を配信するルーティング
@app.route('/assets/<path:filename>')
def serve_assets(filename):
    return send_from_directory(os.path.join(BASE_DIR, 'assets'), filename)

# バックエンド（LLM推論）のAPIルーティング
@app.route('/chat', methods=['POST'])
# def chat(prompt, rag_info=None, system_prompt):
def chat():
    # UIからのリクエストを受け取る
    data = request.json
    user_text = data.get('text', '')

    print(f"\n🐋🐋[Request Received] ユーザー入力: {user_text}")

    # デフォルト値を設定
    persona_type = data.get('persona', 'default')
    # emotion_override = data.get('emotion', 'auto')

    # マネージャーのプロンプトを動的に書き換える
    if persona_type == "tsundere":
        ai_manager.system_prompt = "あなたはツンデレな後輩です。少し怒りっぽく、素直になれない態度で話します。"
    elif persona_type == "cool":
        ai_manager.system_prompt = "あなたは冷徹で論理的な研究者です。感情を交えず、淡々と事実のみを話します。"
    else: # frontendのセレクトボックスで "default" が選ばれた場合は、デフォルトの親しみやすい大学広報の設定に戻す
        ai_manager.system_prompt = "あなたは感情豊かな大学広報の人間です。親しみやすく、丁寧な口調で話します。"

    ### メイン処理
    response_data = ai_manager.generate_reply(
                                                user_text, 
                                                rag_flag=rag_flag # RAGを有効にするかどうかのフラグ、UIから指定できるようにする予定
                                                )
    
    # デバック出力
    print(f'[Input Prompt] {response_data["prompt"]}')
    print(f'[Token Info] {response_data["token_info"]}')

    # UIへ結果を返す
    return jsonify({
        "reply": response_data["reply"],
        "emotion": response_data["complex_emotion"],
        "emotion_state": response_data["emotion_state"],
        # "audio_url": audio_url # 音声ファイルのパスをフロントへ渡す
    })
    

# 音声合成だけを担当する専用API
@app.route('/synthesize', methods=['POST'])
def synthesize():
    data = request.json
    text = data.get('text', '')
    
    # フロントエンドから送られてきた speaker id を渡す
    speaker_id = data.get('speaker_id', 8) # デフォルトは8(春日部つむぎ)
    
    if not text:
        return jsonify({"audio_url": None})

    audio_filename = f"response_{int(time.time())}.wav"
    # audio_url = voicevox.generate_audio(text, output_filename=audio_filename)
    audio_url = voicevox.generate_audio(
        text,
        speaker_id=speaker_id,
        output_filename=audio_filename
    )

    return jsonify({"audio_url": audio_url})

### ログ用のエンドポイント
# Python側：ログをキューに貯める
log_queue = queue.Queue()

# 既存の print の代わりに使うラッパー
def log(msg):
    print(msg)           # Colabのセル出力にも残す
    log_queue.put(msg)   # UIにも流す

# SSE（Server-Sent Events）エンドポイント
@app.route('/logs')
def stream_logs():
    def generate():
        while True:
            try:
                msg = log_queue.get(timeout=1)
                yield f"data: {msg}\n\n"
            except queue.Empty:
                yield f"data: \n\n"  # keepalive
    return Response(generate(), mimetype='text/event-stream')

# タイトルに戻ったら感情をリセットするための処理
@app.route('/reset', methods=['POST'])
def reset_emotion():
    ai_manager.brain.state = np.zeros(ai_manager.brain.num_emotions)
    return jsonify({"status": "ok"})

### 観測所

In [ ]:
### サーバとcloudflare tunnel の起動
import threading
import subprocess

def run_app():
    # host='0.0.0.0' を追加（Colab内のすべての通信を許可するおまじない）
    app.run(host='0.0.0.0', port=8000, use_reloader=False)


# flaskを別スレッドで起動してcolabのセルをブロックしないようにする
threading.Thread(target=run_app, daemon=True).start()

print("🫠 Starting cloudflared tunnel...")
process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# 発行されたURLを正規表現で確実に抽出して表示
for line in process.stdout:
    # "https://〜.trycloudflare.com" のパターンを検索
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        print(f"\n😁 Opend server! Please access below URL:")
        print(f"👉: {url}")
        break